In [ ]:
Grisha Nagpal
e23cseu0036

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import gzip

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
file_path = "facebook_combined.txt.gz"

edges = []
with gzip.open(file_path, 'rt') as f:
    for line in f:
        u, v = map(int, line.split())
        edges.append((u, v))

df = pd.DataFrame(edges, columns=["source", "target"])

print("Dataset Loaded:")
print(df.head())

Dataset Loaded:
   source  target
0       0       1
1       0       2
2       0       3
3       0       4
4       0       5


In [3]:
G = nx.from_pandas_edgelist(df, "source", "target")

In [4]:
degrees = dict(G.degree())

data = pd.DataFrame({
    "node": list(degrees.keys()),
    "degree": list(degrees.values())
})

In [5]:
threshold = np.median(data["degree"])
data["label"] = (data["degree"] > threshold).astype(int)

print("\nSample Data:")
print(data.head())


Sample Data:
   node  degree  label
0     0     347      1
1     1      17      0
2     2      10      0
3     3      17      0
4     4      10      0


In [6]:
X = data[["degree"]]
y = data["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


Build MLP Model

In [7]:
model = Sequential([
    Dense(16, activation='relu', input_shape=(1,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8319 - loss: 0.4861 - val_accuracy: 0.8181 - val_loss: 0.4127
Epoch 2/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8821 - loss: 0.3253 - val_accuracy: 0.9072 - val_loss: 0.2733
Epoch 3/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9452 - loss: 0.2000 - val_accuracy: 0.9592 - val_loss: 0.1659
Epoch 4/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9715 - loss: 0.1235 - val_accuracy: 0.9592 - val_loss: 0.1140
Epoch 5/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9799 - loss: 0.0866 - val_accuracy: 0.9827 - val_loss: 0.0852
Epoch 6/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9861 - loss: 0.0681 - val_accuracy: 0.9827 - val_loss: 0.0700
Epoch 7/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9916 - loss: 0.0569 - val_accuracy: 0.9827 - val_loss: 0.0604
Epoch 8/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9929 - loss: 0.0487 - val_accuracy: 0.

In [9]:
loss, accuracy = model.evaluate(X_test, y_test)
print("\nTest Accuracy:", accuracy)

# Step 10: Predictions
y_pred = (model.predict(X_test) > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0261

Test Accuracy: 1.0
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       420
           1       1.00      1.00      1.00       388

    accuracy                           1.00       808
   macro avg       1.00      1.00      1.00       808
weighted avg       1.00      1.00      1.00       808

